# Evaluation and visualization

Compare persistence and Kalman forecast errors with true RMSE on one explicit split. These tables come from saved results; they are not a new run.

In [ ]:
from pathlib import Path
import pandas as pd
import tomllib
import matplotlib.pyplot as plt

from storm_tracks.evaluation import (
    relative_rmse_improvement,
    results_metadata,
    summarize_lead_time_errors,
)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
with (PROJECT_ROOT / "configs" / "pipeline.toml").open("rb") as file:
    config = tomllib.load(file)
split_name = config["splits"]["evaluation"]
results_dir = PROJECT_ROOT / config["paths"]["results_dir"]
if split_name == "test":
    persistence_path = results_dir / "null_model_test_results.pkl"
    kalman_path = results_dir / "sliding_results_final.pkl"
else:
    persistence_path = results_dir / "null_model_val_results.pkl"
    kalman_path = results_dir / "val_sliding_results_final.pkl"
persistence_results = pd.read_pickle(persistence_path)
kalman_results = pd.read_pickle(kalman_path)

## Summarize errors by forecast horizon

In [ ]:
persistence_summary = summarize_lead_time_errors(
    persistence_results, split=split_name
)
kalman_summary = summarize_lead_time_errors(kalman_results, split=split_name)
comparison = persistence_summary[["rmse"]].rename(
    columns={"rmse": "persistence_rmse"}
).join(kalman_summary[["rmse"]].rename(columns={"rmse": "kalman_rmse"}))
comparison

## Calculate the overall improvement

In [ ]:
improvement_pct = relative_rmse_improvement(
    baseline_errors=persistence_results["error_km"],
    candidate_errors=kalman_results["error_km"],
)
print(f"Overall RMSE improvement on {split_name}: {improvement_pct:.1f}%")
results_metadata(split=split_name)

## Plot RMSE by forecast horizon

In [ ]:
axis = comparison.plot(marker="o", ylabel="RMSE (km)", xlabel="Lead time (hours)")
axis.set_title(f"Forecast RMSE on the {split_name} split")
axis.grid(alpha=0.25)
plt.show()

## Reading the results

Keep the split metadata with every reported metric. Dashboard tables should be regenerated only after the validated data contract is available.